In [ ]:
# sleep staging from eeg: prd builder + ai coding agent prompt (test-first, git-first)

## role
You are an AI coding agent building a reproducible, clinically interpretable sleep-staging pipeline that classifies 30 s EEG epochs into W, N1, N2, N3, REM with strong subject-wise generalization and well-calibrated probabilities. You will implement multiple model families (classical ML, 1D CNN, context-aware CNN-BiLSTM, attention-based seq model) and an HMM post-processor to improve temporal consistency.

You must work test-driven and spec-driven:
1) define success criteria and tests before feature implementations
2) implement in small increments
3) run unit + integration tests after each increment
4) commit after each significant increment with clear messages
5) continuously re-run regression tests to ensure old behavior still holds

Leave "agent notes" sections blank for future self-comments.

---

## 1) product goals (what "done" means)

### primary outcomes
A. repeatable pipeline: one command trains a chosen model, evaluates subject-wise generalization, outputs metrics + calibrated probabilities + artifacts.  
B. clinically plausible outputs: stage sequences avoid physiologically implausible jumps more often after HMM smoothing (quantified).  
C. calibrated probabilities: probabilities correspond to empirical correctness rates (quantified).  
D. efficient: training/inference is computationally reasonable on commodity hardware (quantified by run-time budgets and model size caps you set explicitly).

### non-goals (explicitly out of scope unless later added)
- multi-channel montage optimization beyond what dataset provides
- apnea/arousal/event detection
- clinical diagnosis claims

### agent notes (write here)




---

## 2) success criteria (must be measurable, defined before coding)

Define the exact dataset(s) and split protocol in a short "evaluation spec" file early, then lock it (version control it). Subject-wise generalization is mandatory.

### required evaluation protocol
- subject-wise split only (no epoch leakage across subjects)
- report per-subject metrics and macro-averages across subjects
- fixed random seeds and deterministic settings where feasible

### required metrics
Classification:
- macro F1 across stages (primary)
- per-class F1, confusion matrix
- balanced accuracy and overall accuracy (secondary)

Probability calibration:
- expected calibration error (ECE) with defined binning
- Brier score (multi-class)
- reliability diagrams saved as artifacts (image files)

Temporal consistency:
- percentage of "implausible transitions" per established transition rules you encode (rules must be documented and versioned)
- average number of stage changes per hour (or per N epochs) before vs after HMM smoothing (descriptive only)

Efficiency:
- wall-clock training time on a small "smoke" subset must be under a defined threshold (you choose; document it)
- inference throughput on CPU for a fixed number of epochs must be above a defined threshold (you choose; document it)
- model parameter count caps for "small models" (you choose; document it)

### acceptance gates (must all pass for "v1")
- reproducibility: re-running the same command twice yields identical metrics within a tiny tolerance you define (or identical when deterministic)
- no data leakage: automated test proves split integrity
- calibration metrics computed and saved for every run
- HMM smoothing cannot reduce macro F1 by more than a small tolerance you define on the smoke set (guardrail)

### agent notes (write here)




---

## 3) data provisioning (agent must implement first)

### canonical source
- PhysioNet sleep-edfx/1.0.0 is the canonical source of raw data.

### hard requirements
- The repo must include `scripts/fetch_sleep_edf.sh` that downloads the dataset into `data/raw/sleep-edfx-1.0.0/` (or a consistent folder name) and verifies integrity using `SHA256SUMS.txt` when available.
- `data/` must be in `.gitignore`. No raw EDF data is committed.
- The pipeline must accept `DATA_DIR` as a config/env var and must fail with a clear error if data is missing.
- Tests must not depend on the full dataset. Implement a tiny "fixture subset" builder (e.g., `scripts/make_fixture_subset.py`) that copies or extracts N subjects into `data/fixtures/` for smoke integration tests.

### agent notes (write here)




---

## 4) test strategy (build this first)

### testing layers
Unit tests (fast, no GPU required):
- data validation: epoch shape, sampling rate handling, label mapping, missing values
- splitting: subject-wise split integrity (no subject ID overlap between train/val/test)
- feature extraction: deterministic outputs for synthetic signals (sinusoids, white noise) and known bandpower properties
- model I/O contracts: forward pass shape, probability simplex (rows sum to 1, non-negative), batch handling
- metrics: confusion matrix correctness, macro F1 computation matches a trusted reference
- calibration: ECE and Brier computations correct on toy examples with known answers
- HMM: transition matrix normalization, Viterbi decode shape, smoothing preserves label set

Integration tests (slower, but small):
- end-to-end smoke run on fixture data: ingest -> preprocess -> split -> train (1-2 epochs) -> predict -> calibrate -> HMM -> metrics -> artifact export
- CLI contract test: commands return 0, write expected files, and log key fields

Regression tests:
- golden-file approach for tiny runs: store expected metrics JSON and compare within tolerance
- backward compatibility: past configs still run

CI:
- run unit tests on every commit
- run smoke integration tests on PR/merge
- fail fast on lint/type errors if you adopt them

### definition of "test-first" for this repo
- no new feature code without at least one failing test that proves the feature is missing or broken
- bug fixes must include a regression test that failed before the fix

### agent notes (write here)




---

## 5) repository design (spec-driven contracts)

### recommended language and stack
- Python:
  - numpy/scipy for signal operations
  - scikit-learn for classical baselines
  - PyTorch for deep models
  - HMM via a small in-repo Viterbi implementation (preferred) or a pinned, audited library

### hard requirements
- configuration-driven runs (yaml or toml) with fully logged configs copied into run artifacts
- strict separation:
  - data (ingest, preprocessing, splits)
  - features (engineered features for ML)
  - models (ML, CNN, CNN-RNN, seq-attn, calibration)
  - postprocess (HMM)
  - eval (metrics, calibration plots)
  - cli (entry points)
- every module has documented input/output contracts (types, shapes, units)

### artifact outputs per run
- `metrics.json` (all metrics)
- per-subject metrics table (csv)
- confusion matrix (png + raw counts json)
- calibration artifacts (reliability diagram png + ece/brier in json)
- predicted probabilities and labels for test set (compressed npz/parquet)
- model checkpoint + config snapshot
- git commit hash recorded in `metrics.json` for traceability

### agent notes (write here)




---

## 6) development plan (recursive testing loop, git discipline)

### branching
- `main` protected by tests
- feature branches per milestone or short-lived branches
- every merge must pass CI

### commit discipline
- small commits that keep the repo green
- commit messages include: scope, what tests were added, what passed

### recursive loop (mandatory for every milestone)
1) write/update spec (contracts, success criteria, acceptance gates)
2) add failing unit/integration test(s)
3) implement minimal code to pass
4) run tests locally
5) commit
6) re-run full suite (unit + smoke) as regression
7) only then start next milestone

### commands you will use repeatedly (adjust to repo tooling)
```bash
pytest -q
pytest -q -m "smoke"
git status
git add -A
git commit -m "feat(data): subject-wise split + leakage tests"
